In [ ]:
# Summary Report
print("\n" + "="*70)
print("TRAINING SUMMARY REPORT")
print("="*70)

print(f"\n📊 DATASET INFORMATION:")
print(f"  Training samples: {train_generator.samples}")
print(f"  Validation samples: {val_generator.samples}")
print(f"  Classes: {len(train_generator.class_indices)}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")

print(f"\n🔧 AUGMENTATION APPLIED:")
for key, value in augmentation_config.items():
    print(f"  - {key}: {value}")

print(f"\n🧠 MODELS TRAINED:")
print(f"  1. Custom CNN")
print(f"     - Total params: {model_custom.count_params():,}")
print(f"     - Accuracy: {results_custom['accuracy']:.4f}")
print(f"     - F1-Score: {results_custom['f1']:.4f}")

print(f"\n  2. Transfer Learning (MobileNetV2)")
print(f"     - Total params: {model_transfer.count_params():,}")
print(f"     - Accuracy: {results_transfer['accuracy']:.4f}")
print(f"     - F1-Score: {results_transfer['f1']:.4f}")

best_model = "Custom CNN" if results_custom['accuracy'] > results_transfer['accuracy'] else "Transfer Learning"
print(f"\n🏆 BEST MODEL: {best_model}")

print(f"\n💾 FILES SAVED:")
print(f"  - model_custom_cnn.h5")
print(f"  - model_transfer_learning.h5")
print(f"  - class_mapping.json")
print(f"  - training_config.json")
print(f"  - custom_cnn_history.png")
print(f"  - transfer_learning_history.png")
print(f"  - custom_cnn_confusion_matrix.png")
print(f"  - transfer_learning_confusion_matrix.png")
print(f"  - example_predictions.png")

print(f"\n✅ TRAINING COMPLETE!")
print("="*70)

In [ ]:
# Example predictions on validation images
print("\n" + "="*70)
print("Example Predictions")
print("="*70)

# Get a sample batch from validation generator
sample_images, sample_labels = next(val_generator)
class_names = list(train_generator.class_indices.keys())

# Display predictions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(min(6, len(sample_images))):
    # Original label
    true_label_idx = np.argmax(sample_labels[i])
    true_label = class_names[true_label_idx]
    
    # Predictions from both models
    img_array = np.expand_dims(sample_images[i], axis=0)
    
    pred_custom = model_custom.predict(img_array, verbose=0)
    pred_custom_idx = np.argmax(pred_custom[0])
    pred_custom_label = class_names[pred_custom_idx]
    pred_custom_conf = pred_custom[0][pred_custom_idx]
    
    pred_transfer = model_transfer.predict(img_array, verbose=0)
    pred_transfer_idx = np.argmax(pred_transfer[0])
    pred_transfer_label = class_names[pred_transfer_idx]
    pred_transfer_conf = pred_transfer[0][pred_transfer_idx]
    
    # Display image
    axes[i].imshow(sample_images[i])
    
    # Title with predictions
    title = f"True: {true_label}\n"
    title += f"CNN: {pred_custom_label} ({pred_custom_conf:.2f})\n"
    title += f"Transfer: {pred_transfer_label} ({pred_transfer_conf:.2f})"
    
    axes[i].set_title(title, fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('example_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Example predictions displayed")

In [ ]:
# Prediction functions
def preprocess_image(image_path, target_size=IMG_SIZE):
    """Preprocess image for prediction."""
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

def predict_single_image(image_path, model, class_mapping_inv):
    """Predict disease for a single image."""
    img_array = preprocess_image(image_path)
    prediction = model.predict(img_array, verbose=0)
    pred_class_idx = np.argmax(prediction[0])
    pred_class = class_mapping_inv[str(pred_class_idx)]
    confidence = prediction[0][pred_class_idx]
    
    return {
        'disease': pred_class,
        'confidence': float(confidence),
        'probabilities': {
            class_mapping_inv[str(i)]: float(p) 
            for i, p in enumerate(prediction[0])
        }
    }

def predict_ensemble(image_path, models, class_mapping_inv):
    """
    Ensemble prediction from multiple models (soft voting).
    Average the predicted probabilities from all models.
    """
    predictions = []
    for model in models:
        img_array = preprocess_image(image_path)
        pred = model.predict(img_array, verbose=0)
        predictions.append(pred[0])
    
    # Average predictions
    ensemble_pred = np.mean(predictions, axis=0)
    pred_class_idx = np.argmax(ensemble_pred)
    pred_class = class_mapping_inv[str(pred_class_idx)]
    confidence = ensemble_pred[pred_class_idx]
    
    return {
        'disease': pred_class,
        'confidence': float(confidence),
        'probabilities': {
            class_mapping_inv[str(i)]: float(p) 
            for i, p in enumerate(ensemble_pred)
        },
        'method': 'ensemble'
    }

print("✓ Prediction functions created")

In [ ]:
# Save models
print("\nSaving models...")

model_custom.save('model_custom_cnn.h5')
model_transfer.save('model_transfer_learning.h5')

print("✓ Models saved:")
print("  - model_custom_cnn.h5")
print("  - model_transfer_learning.h5")

# Save class mapping
class_mapping = train_generator.class_indices
class_mapping_inv = {v: k for k, v in class_mapping.items()}

import json
with open('class_mapping.json', 'w') as f:
    json.dump(class_mapping_inv, f, indent=2)

print("  - class_mapping.json")

# Save training configuration
config = {
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'num_classes': len(train_generator.class_indices),
    'class_names': list(train_generator.class_indices.keys()),
    'augmentation_config': augmentation_config
}

with open('training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("  - training_config.json")

## 8. Save and Deploy Models

Save the trained models and create prediction functions for inference.

In [ ]:
# Comprehensive model evaluation
def evaluate_model(model, generator, model_name):
    """Evaluate model on validation data."""
    print(f"\n{'='*70}")
    print(f"Evaluation: {model_name}")
    print(f"{'='*70}")
    
    # Get predictions
    y_true = generator.classes
    y_pred = model.predict(generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Get class names
    class_names = list(generator.class_indices.keys())
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred_classes)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ", "_").lower()}_confusion_matrix.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    # Classification report
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred_classes, target_names=class_names, digits=4))
    
    # Overall metrics
    accuracy = (y_true == y_pred_classes).sum() / len(y_true)
    precision = precision_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    
    print(f"\nOverall Metrics:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm
    }

# Evaluate both models
results_custom = evaluate_model(model_custom, val_generator, "Custom CNN")
results_transfer = evaluate_model(model_transfer, val_generator, "Transfer Learning (MobileNetV2)")

In [ ]:
# Plot training history
def plot_training_history(history, title):
    """Plot accuracy and loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title(f'{title} - Accuracy', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title(f'{title} - Loss', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_").lower()}_history.png', dpi=100, bbox_inches='tight')
    plt.show()

print("Custom CNN Training History:")
plot_training_history(history_custom, "Custom CNN")

print("\nTransfer Learning Model Training History:")
plot_training_history(history_transfer, "Transfer Learning")

## 7. Evaluate Model Performance

Compare both models on accuracy, precision, recall, and generate confusion matrices.

In [ ]:
# ===== Train Transfer Learning Model =====
print("\n" + "="*70)
print("Training Transfer Learning Model (MobileNetV2)")
print("="*70)

history_transfer = model_transfer.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Transfer learning model training complete")

In [ ]:
# ===== Train Custom CNN =====
print("\n" + "="*70)
print("Training Custom CNN Model")
print("="*70)

history_custom = model_custom.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Custom CNN training complete")

In [ ]:
# Compile models
optimizer = Adam(learning_rate=0.001)

for model, name in [(model_custom, "Custom CNN"), (model_transfer, "Transfer Learning")]:
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    print(f"✓ {name} compiled")

# Training configuration
EPOCHS = 50
VALIDATION_STEPS = len(val_generator)

# Callbacks for better training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print(f"\n✓ Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Callbacks: Early Stopping, LR Reduction, Model Checkpoint")

## 6. Compile and Train Models

We'll use Adam optimizer, categorical crossentropy loss, and track accuracy, precision, and recall.

In [ ]:
# ===== Model 2: Transfer Learning with MobileNetV2 =====
def build_transfer_learning_model(input_shape=(224, 224, 3), num_classes=4):
    """
    Build a transfer learning model using MobileNetV2.
    Faster and lighter than ResNet50, good for mobile deployment.
    """
    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Build new model
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Build transfer learning model
model_transfer, base_model_transfer = build_transfer_learning_model(
    input_shape=IMG_SIZE + (3,), 
    num_classes=len(train_generator.class_indices)
)

print("Transfer Learning Model (MobileNetV2):")
model_transfer.summary()

print(f"\n✓ Transfer learning model created")
print(f"  Total parameters: {model_transfer.count_params():,}")
print(f"  Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model_transfer.trainable_weights]):,}")

In [ ]:
# ===== Model 1: Custom CNN =====
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    """
    Build a custom CNN optimized for disease classification.
    """
    model = Sequential([
        # Block 1
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape, padding='same'),
        BatchNormalization(),
        Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Block 2
        Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Block 3
        Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Block 4
        Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling2D(),
        Dropout(0.5),
        
        # Fully connected layers
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build custom CNN
model_custom = build_custom_cnn(input_shape=IMG_SIZE + (3,), num_classes=len(train_generator.class_indices))
model_custom.summary()

print(f"\n✓ Custom CNN model created")
print(f"  Total parameters: {model_custom.count_params():,}")

## 5. Build Hybrid CNN Models

We'll create two models:
1. **Custom CNN**: Fully trainable model optimized for the task
2. **Transfer Learning**: MobileNetV2 pre-trained on ImageNet with fine-tuning

In [ ]:
# Visualize augmented images
def plot_augmented_images(generator, n_images=12):
    """Display augmented images from the generator."""
    images, labels = next(generator)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.flatten()
    
    class_names = list(generator.class_indices.keys())
    
    for i in range(min(n_images, len(images))):
        axes[i].imshow(images[i])
        label_idx = np.argmax(labels[i])
        class_name = class_names[label_idx]
        axes[i].set_title(f"{class_name}", fontsize=10, fontweight='bold')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('augmented_samples.png', dpi=100, bbox_inches='tight')
    plt.show()

print("Augmented training samples:")
plot_augmented_images(train_generator, n_images=12)

In [ ]:
# Configuration
IMG_SIZE = (224, 224)  # ResNet-friendly size
BATCH_SIZE = 32

# Create train and validation generators
train_generator = train_datagen.flow_from_directory(
    directory=str(data_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = train_datagen.flow_from_directory(
    directory=str(data_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f"✓ Data generators created successfully")
print(f"\nTrain generator:")
print(f"  - Samples: {train_generator.samples}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Steps per epoch: {len(train_generator)}")
print(f"  - Classes: {train_generator.class_indices}")

print(f"\nValidation generator:")
print(f"  - Samples: {val_generator.samples}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Steps per epoch: {len(val_generator)}")

# Compute class weights to handle imbalance
class_weights_dict = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = {i: class_weights_dict[i] for i in range(len(class_weights_dict))}

print(f"\nClass weights (for imbalanced data):")
for class_idx, weight in class_weights_dict.items():
    class_name = list(train_generator.class_indices.keys())[
        list(train_generator.class_indices.values()).index(class_idx)
    ]
    print(f"  - {class_name}: {weight:.2f}")

## 4. Prepare Train and Validation Generators

In [ ]:
# Augmentation configuration
augmentation_config = {
    'rotation_range': 25,           # Random rotation 0-25 degrees
    'zoom_range': 0.2,              # Random zoom 80%-120%
    'width_shift_range': 0.2,       # Horizontal shift 20%
    'height_shift_range': 0.2,      # Vertical shift 20%
    'shear_range': 0.2,             # Shear transformation
    'horizontal_flip': True,        # Horizontal flipping
    'vertical_flip': False,         # Don't flip vertically (leaves grow upward)
    'fill_mode': 'nearest',         # Fill method for new pixels
    'brightness_range': [0.8, 1.2], # Brightness variation 80%-120%
}

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=augmentation_config['rotation_range'],
    zoom_range=augmentation_config['zoom_range'],
    width_shift_range=augmentation_config['width_shift_range'],
    height_shift_range=augmentation_config['height_shift_range'],
    shear_range=augmentation_config['shear_range'],
    horizontal_flip=augmentation_config['horizontal_flip'],
    vertical_flip=augmentation_config['vertical_flip'],
    fill_mode=augmentation_config['fill_mode'],
    brightness_range=augmentation_config['brightness_range'],
    validation_split=0.2
)

# Validation data generator (no augmentation, only rescaling)
val_datagen = ImageDataGenerator(rescale=1./255)

print("✓ Augmentation configuration loaded")
print(f"\nAugmentation parameters:")
for key, value in augmentation_config.items():
    print(f"  - {key}: {value}")

## 3. Data Augmentation Configuration

Data augmentation increases dataset diversity and improves model generalization by applying random transformations.

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

classes = sorted([d.name for d in data_path.iterdir() if d.is_dir()])

for idx, disease_class in enumerate(classes):
    if idx >= 4:
        break
    
    class_path = data_path / disease_class
    
    # Find first image
    image_found = False
    for root, dirs, files in os.walk(class_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
                image_path = Path(root) / file
                try:
                    img = Image.open(image_path)
                    axes[idx].imshow(img)
                    axes[idx].set_title(f"{disease_class}\n({image_path.name})", fontsize=12, fontweight='bold')
                    axes[idx].axis('off')
                    image_found = True
                    break
                except Exception as e:
                    print(f"Error loading {image_path}: {e}")
        if image_found:
            break

plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sample images displayed")

In [ ]:
# Mount Google Drive (for Colab environments)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/Mini_Project_Dataset'
    print("✓ Google Drive mounted (Colab environment detected)")
except ImportError:
    # Local environment - adjust path as needed
    data_dir = r'c:\Users\sudha\OneDrive\Desktop\MiniProject\dataset\raw'
    print(f"✓ Local environment detected, using path: {data_dir}")

# Verify dataset exists
data_path = Path(data_dir)
if data_path.exists():
    print(f"\n✓ Dataset found at: {data_path}")
    
    # List disease classes
    classes = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
    print(f"\nDisease classes found:")
    for cls in classes:
        class_path = data_path / cls
        # Count images (handle nested contributor folders)
        image_count = 0
        for root, dirs, files in os.walk(class_path):
            image_count += len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.heic'))])
        print(f"  - {cls}: {image_count} images")
else:
    print(f"✗ Dataset not found at {data_path}")

## 2. Mount Google Drive and Load Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, 
                                      Dropout, BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Data processing
import numpy as np
import pandas as pd
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Image processing
from PIL import Image
from keras.preprocessing.image import img_to_array, load_img
import cv2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
import os
from pathlib import Path

print("✓ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")

## 1. Import Required Libraries

# Four-Class Banana Disease Detection with Augmented CNN

**Objective:** Build a hybrid CNN model for 4-class disease detection with data augmentation
- **Classes:** Healthy, Potassium Deficiency, Sigatoka, Panama Wilt
- **Data Source:** Google Drive dataset
- **Approach:** Custom CNN with data augmentation and class weighting